In [ ]:
import gymnasium as gym
import planar_quadcopter_env
from gymnasium.wrappers import ClipAction, RecordVideo
from supersuit import nan_zeros_v0
import numpy as np
num_episodes = 1
quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode=None, max_episode_steps=120)
# quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode="rgb_array", max_episode_steps=120)
# quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode="human", max_episode_steps=120, noisy=True)
env = nan_zeros_v0(ClipAction(quad_env))
# env = RecordVideo(env, video_folder="videos", episode_trigger=lambda t: t%10==0, disable_logger=True)
# print(env)
trajs = []
trajs_controls = []
options = [
    {
        "low": np.array([0,0, 0, 0, 0, 0]),
        "high": np.array([0,0, 0, 0, 0, 0])
    },
    {
        "low": np.array([5-1e-4,6-1e-4, 0.285, 1, 1, 1]),
        "high": np.array([5+1e-4,6+1e-4, 0.285, 1, 1, 1])
    },
    {
        "low": np.array([2,-6, 0.585, 0, 0, 0]),
        "high": np.array([2,-6, 0.585, 0, 0, 0])
    },
    {
        "low": np.array([7,0, 0, 3, 2, 0]),
        "high": np.array([7,0, 0, 3, 2, 0])
    },
]
for i in range(num_episodes):
    # new_state, _ = env.reset()
    # new_state, _ = env.reset(options={
    #     "low": np.array([20,20,-np.pi/8, 0, 0, 0]),
    #     "high": np.array([35,35, np.pi/8, 0, 0, 0])
    # })
    new_state, _ = env.reset(options=options[0])
    state_trajectories = [new_state]
    controls = [0.03,-0.1]

    while True:
        # Do a control
        # Implement a controller
        new_state,_,terminated,truncated,_ = env.step(controls)
        state_trajectories.append(new_state)
        # print(type(new_state))
        if terminated or truncated:
            break

    trajs.append(np.array(state_trajectories))
    trajs_controls.append(np.array(controls))
env.close()

# RRT Planning Noiseless


In [ ]:
import matplotlib.pyplot as plt
from rrt import RRT

goal_center, goal_radius = np.array([8.0, 8.0]), 3.0
obstacle_radius = 4.0  # obstacle: circle at the origin

is_goal = lambda s: np.linalg.norm(s[:2] - goal_center) < goal_radius
is_free = lambda s: np.linalg.norm(s[:2]) >= obstacle_radius and not is_goal(s)

x0 = np.array([-9.0, -9.0, 0, 0, 0, 0])
rrt_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=25, render_mode=None)

rrt = RRT(rrt_env, x0, is_free, is_goal, max_iter=30000, seed=0)  # ~2 min
rrt.plan()

In [ ]:
ALL_PATHS = True  # True: every goal path. False: only paths of the most common length.

if ALL_PATHS:
    paths = rrt.goal_paths()
    title = f"all {len(paths)} goal paths"
else:
    T, paths = rrt.most_common_length_paths()
    title = f"{len(paths)} goal paths of length T={T}"

fig, (ax_tree, ax_paths) = plt.subplots(1, 2, figsize=(14, 7))

# Plot the RRT
edges = np.array([[n.parent.state[:2], n.state[:2], [np.nan, np.nan]] for n in rrt.nodes[1:]])
ax_tree.plot(edges[:, :, 0].ravel(), edges[:, :, 1].ravel(), 'C0', lw=0.3)
ax_tree.set_title(f"RRT: {len(rrt.nodes)} nodes")

# Plot the returned paths.
for states, controls in paths:
    ax_paths.plot(states[:, 0], states[:, 1], lw=0.8)
ax_paths.set_title(title)

for ax in (ax_tree, ax_paths):
    ax.plot(*x0[:2], 'k*', ms=12)
    ax.set_xlim(-12.5, 12.5); ax.set_ylim(-12.5, 12.5)
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.show()